# 08 - YOLOv8s Training & Evaluation for Retail Theft Detection

**Purpose:** Train YOLOv8s model and evaluate performance

**Objectives:**
- Verify GPU availability (CRITICAL)
- Train YOLOv8s on retail theft dataset
- Monitor training progress
- Evaluate model performance
- Analyze results with focus on theft class recall
- Export trained model

---

## 1. Setup and GPU Verification (CRITICAL)

In [1]:
# Install required packages
!pip install ultralytics torch torchvision pyyaml pandas matplotlib seaborn --quiet

In [2]:
import os
import sys
import yaml
import json
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')

print("Basic imports successful!")

Basic imports successful!


In [3]:
# CRITICAL: GPU Verification
import torch

print("="*70)
print("GPU VERIFICATION (CRITICAL)")
print("="*70)

if not torch.cuda.is_available():
    print("\n" + "!"*70)
    print("!! CRITICAL WARNING: NO GPU DETECTED !!")
    print("!"*70)
    print("""
Training on CPU is NOT recommended for YOLOv8.
It will be extremely slow (10-100x slower than GPU).

Please ensure:
1. NVIDIA GPU is installed and detected
2. CUDA toolkit is installed (version 11.8 or 12.x)
3. PyTorch is installed with CUDA support:
   pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118

Check NVIDIA driver: nvidia-smi
    """)
    USE_GPU = False
    DEVICE = 'cpu'
else:
    print(f"\n[SUCCESS] GPU DETECTED!")
    print(f"\nPyTorch version: {torch.__version__}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"cuDNN version: {torch.backends.cudnn.version()}")
    
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\nGPU {i}: {props.name}")
        print(f"  Memory: {props.total_memory / 1024**3:.1f} GB")
        print(f"  Compute Capability: {props.major}.{props.minor}")
    
    USE_GPU = True
    DEVICE = 0  # Use first GPU

print(f"\nTraining device: {DEVICE}")
print("="*70)

GPU VERIFICATION (CRITICAL)

[SUCCESS] GPU DETECTED!

PyTorch version: 2.9.1+cu130
CUDA version: 13.0
cuDNN version: 91200

GPU 0: NVIDIA GeForce GTX 1650
  Memory: 4.0 GB
  Compute Capability: 7.5

Training device: 0


In [4]:
# Import Ultralytics YOLO
from ultralytics import YOLO
import ultralytics

print(f"Ultralytics version: {ultralytics.__version__}")
print("YOLO imported successfully!")

Ultralytics version: 8.3.247
YOLO imported successfully!


In [5]:
# Configuration
BASE_DIR = Path(r"c:/Users/shaho/OneDrive - Nile University/Desktop/AletrixGrad")

# Dataset directory (use augmented if available)
AUGMENTED_DIR = BASE_DIR / "dataset_augmented"
CLEANED_DIR = BASE_DIR / "dataset_cleaned"
ORIGINAL_DIR = BASE_DIR / "cc-tv-footage-annotation-b8-lcysc-b1-2"

for dataset_dir in [AUGMENTED_DIR, CLEANED_DIR, ORIGINAL_DIR]:
    if dataset_dir.exists() and (dataset_dir / "train" / "images").exists():
        DATASET_DIR = dataset_dir
        break
else:
    DATASET_DIR = ORIGINAL_DIR

CONFIGS_DIR = BASE_DIR / "configs"
RUNS_DIR = BASE_DIR / "runs"
OUTPUT_DIR = BASE_DIR / "outputs"
VIS_DIR = BASE_DIR / "visualizations"

# Create directories
for d in [CONFIGS_DIR, RUNS_DIR, OUTPUT_DIR, VIS_DIR]:
    d.mkdir(exist_ok=True)

# Class names
CLASS_NAMES = {
    0: 'Customer-Bagpack',
    1: 'Product',
    2: 'Product-Picked',
    3: 'Shopping-Cart',
    4: 'normal',
    5: 'theft'
}

print(f"Dataset: {DATASET_DIR}")
print(f"Runs directory: {RUNS_DIR}")

Dataset: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\dataset_augmented
Runs directory: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\runs


## 2. Prepare Data Configuration

In [6]:
# Create or verify data.yaml
data_yaml_path = CONFIGS_DIR / 'data.yaml'

# Check if data.yaml exists in configs, otherwise create it
if not data_yaml_path.exists():
    data_config = {
        'path': str(DATASET_DIR),
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': 6,
        'names': list(CLASS_NAMES.values())
    }
    
    with open(data_yaml_path, 'w') as f:
        yaml.dump(data_config, f, default_flow_style=False)
    
    print(f"Created data.yaml at: {data_yaml_path}")
else:
    print(f"Using existing data.yaml: {data_yaml_path}")

# Display contents
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)
    
print("\ndata.yaml contents:")
print("-"*40)
print(yaml.dump(data_config, default_flow_style=False))

Using existing data.yaml: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\configs\data.yaml

data.yaml contents:
----------------------------------------
names:
- Customer-Bagpack
- Product
- Product-Picked
- Shopping-Cart
- normal
- theft
nc: 6
path: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\dataset_augmented
test: test/images
train: train/images
val: valid/images



## 3. Training Configuration

In [7]:
# Training hyperparameters
# Adjust batch size based on GPU memory
if USE_GPU:
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if gpu_memory >= 24:
        BATCH_SIZE = 32
    elif gpu_memory >= 16:
        BATCH_SIZE = 16
    elif gpu_memory >= 8:
        BATCH_SIZE = 8
    elif gpu_memory >= 6:
        BATCH_SIZE = 4
    else:
        BATCH_SIZE = 2
else:
    BATCH_SIZE = 4  # Small batch for CPU

# Training parameters
EPOCHS = 100
IMG_SIZE = 640
PATIENCE = 30  # Early stopping

print("="*70)
print("TRAINING CONFIGURATION")
print("="*70)
print(f"\nModel: YOLOv8s (Small)")
print(f"Image Size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Early Stopping Patience: {PATIENCE}")
print(f"Device: {DEVICE}")

TRAINING CONFIGURATION

Model: YOLOv8s (Small)
Image Size: 640x640
Batch Size: 2
Epochs: 100
Early Stopping Patience: 30
Device: 0


## 4. Load Model and Start Training

In [8]:
# Load YOLOv8s pretrained model
print("Loading YOLOv8s model (pretrained on COCO)...")
model = YOLO('yolov8s.pt')
print("Model loaded successfully!")

# Display model architecture summary
print("\nModel Summary:")
print(model.info())

Loading YOLOv8s model (pretrained on COCO)...
Model loaded successfully!

Model Summary:
YOLOv8s summary: 129 layers, 11,166,560 parameters, 0 gradients, 28.8 GFLOPs
(129, 11166560, 0, 28.816844800000002)


In [9]:
# Start Training
print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70)
print(f"\nTime: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Dataset: {DATASET_DIR}")
print(f"Saving results to: {RUNS_DIR}/retail_theft_yolov8s")
print("\nTraining in progress... (this may take several hours)")
print("="*70 + "\n")

# IMPORTANT: On Windows, reduce workers to avoid DataLoader crash
import platform
if platform.system() == 'Windows':
    NUM_WORKERS = 0
    print("[INFO] Windows detected - using 0 workers to avoid DataLoader issues")
else:
    NUM_WORKERS = 8 if USE_GPU else 4

# Train the model
results = model.train(
    data=str(data_yaml_path),
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    device=DEVICE,
    optimizer='AdamW',
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0,
    copy_paste=0.0,
    workers=NUM_WORKERS,
    cache=False,
    amp=False,
    project=str(RUNS_DIR),
    name='retail_theft_yolov8s',
    exist_ok=True,
    pretrained=True,
    verbose=True,
    seed=42,
    val=True,
    plots=True,
    save=True,
    save_period=10,
    patience=PATIENCE,
)

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)
print(f"Results saved to: {results.save_dir}")



STARTING TRAINING

Time: 2026-01-19 03:22:27
Dataset: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\dataset_augmented
Saving results to: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\runs/retail_theft_yolov8s

Training in progress... (this may take several hours)

[INFO] Windows detected - using 0 workers to avoid DataLoader issues
New https://pypi.org/project/ultralytics/8.4.6 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.247  Python-3.10.10 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\configs\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=

## 5. Evaluate Model Performance

In [10]:
# Load best model for evaluation
best_model_path = Path(results.save_dir) / 'weights' / 'best.pt'
print(f"Loading best model from: {best_model_path}")

best_model = YOLO(str(best_model_path))
print("Best model loaded!")

Loading best model from: C:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\runs\retail_theft_yolov8s\weights\best.pt
Best model loaded!


In [12]:
# Validate on validation set
print("\n" + "="*70)
print("VALIDATION RESULTS")
print("="*70)

val_results = best_model.val(
    data=str(data_yaml_path),
    split='val',
    device=DEVICE,
    verbose=True
)

print("\nValidation Metrics:")
print(f"  mAP@0.5:      {val_results.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {val_results.box.map:.4f}")
print(f"  Precision:    {val_results.box.mp:.4f}")
print(f"  Recall:       {val_results.box.mr:.4f}")


VALIDATION RESULTS
Ultralytics 8.3.247  Python-3.10.10 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 914.2242.1 MB/s, size: 122.2 KB)
val: Scanning C:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\dataset_augmented\valid\labels.cache... 482 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 482/482  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 31/31 2.4it/s 13.1s0.4s
                   all        482       1767      0.837      0.779      0.843      0.641
      Customer-Bagpack        129        148      0.827       0.77      0.848      0.537
               Product        188        199      0.774      0.322      0.508      0.267
        Product-Picked        181        192       0.83      0.839      0.879      0.736
         Shopping-Cart         27         27      0.953      0.963      0.971      0.766
                normal        406       

In [13]:
# Validate on test set
print("\n" + "="*70)
print("TEST SET RESULTS")
print("="*70)

test_results = best_model.val(
    data=str(data_yaml_path),
    split='test',
    device=DEVICE,
    verbose=True
)

print("\nTest Set Metrics:")
print(f"  mAP@0.5:      {test_results.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {test_results.box.map:.4f}")
print(f"  Precision:    {test_results.box.mp:.4f}")
print(f"  Recall:       {test_results.box.mr:.4f}")


TEST SET RESULTS
Ultralytics 8.3.247  Python-3.10.10 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 133.870.6 MB/s, size: 167.3 KB)
val: Scanning C:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\dataset_augmented\test\labels... 256 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 256/256 459.1it/s 0.6s0.0s
val: New cache created: C:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\dataset_augmented\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 2.3it/s 7.1s0.4s
                   all        256        842      0.748      0.748      0.779      0.576
      Customer-Bagpack         49         59      0.714      0.746      0.775      0.444
               Product         77         79      0.638      0.313      0.448      0.197
        Product-Picked         78         82      0.661      0.805      0.787      0.628
     

In [14]:
# Per-class metrics
print("\n" + "="*70)
print("PER-CLASS PERFORMANCE")
print("="*70)

class_names = list(CLASS_NAMES.values())

if hasattr(test_results.box, 'ap_class_index') and test_results.box.ap_class_index is not None:
    print("\n{:<20} {:>10} {:>10} {:>10} {:>10}".format(
        'Class', 'Precision', 'Recall', 'mAP@0.5', 'mAP@0.5:0.95'))
    print("-"*60)
    
    # Access per-class metrics
    for i, class_idx in enumerate(test_results.box.ap_class_index):
        class_name = class_names[class_idx] if class_idx < len(class_names) else f'Class {class_idx}'
        p = test_results.box.p[i] if i < len(test_results.box.p) else 0
        r = test_results.box.r[i] if i < len(test_results.box.r) else 0
        ap50 = test_results.box.ap50[i] if i < len(test_results.box.ap50) else 0
        ap = test_results.box.ap[i] if i < len(test_results.box.ap) else 0
        
        print("{:<20} {:>10.4f} {:>10.4f} {:>10.4f} {:>10.4f}".format(
            class_name, p, r, ap50, ap))
        
        # Highlight theft class
        if class_name == 'theft':
            print(f"\n  >>> THEFT CLASS RECALL: {r:.4f} <<<")
            if r < 0.7:
                print("  [WARNING] Theft recall is below 70%. Consider:")
                print("    - More augmentation on theft samples")
                print("    - Lower confidence threshold for inference")
                print("    - Fine-tuning with higher theft class weight")
else:
    print("\nPer-class metrics not available. Check validation results.")


PER-CLASS PERFORMANCE

Class                 Precision     Recall    mAP@0.5 mAP@0.5:0.95
------------------------------------------------------------
Customer-Bagpack         0.7136     0.7458     0.7752     0.4437
Product                  0.6384     0.3129     0.4481     0.1967
Product-Picked           0.6610     0.8049     0.7872     0.6279
Shopping-Cart            0.8449     0.8611     0.8501     0.6973
normal                   0.9081     0.9271     0.9630     0.7879
theft                    0.7209     0.8333     0.8482     0.7049

  >>> THEFT CLASS RECALL: 0.8333 <<<


## 6. Visualize Training Results

In [15]:
# Load training results CSV
results_csv = Path(results.save_dir) / 'results.csv'

if results_csv.exists():
    training_df = pd.read_csv(results_csv)
    training_df.columns = training_df.columns.str.strip()  # Clean column names
    
    print("Training results loaded!")
    print(f"Columns: {list(training_df.columns)}")
else:
    print(f"Results CSV not found at: {results_csv}")
    training_df = None

Training results loaded!
Columns: ['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']


In [16]:
# Plot training curves
if training_df is not None:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Loss curves
    if 'train/box_loss' in training_df.columns:
        axes[0, 0].plot(training_df['epoch'], training_df['train/box_loss'], label='Train', color='blue')
        if 'val/box_loss' in training_df.columns:
            axes[0, 0].plot(training_df['epoch'], training_df['val/box_loss'], label='Val', color='red')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Box Loss')
        axes[0, 0].set_title('Box Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(True)
    
    if 'train/cls_loss' in training_df.columns:
        axes[0, 1].plot(training_df['epoch'], training_df['train/cls_loss'], label='Train', color='blue')
        if 'val/cls_loss' in training_df.columns:
            axes[0, 1].plot(training_df['epoch'], training_df['val/cls_loss'], label='Val', color='red')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Classification Loss')
        axes[0, 1].set_title('Classification Loss')
        axes[0, 1].legend()
        axes[0, 1].grid(True)
    
    if 'train/dfl_loss' in training_df.columns:
        axes[0, 2].plot(training_df['epoch'], training_df['train/dfl_loss'], label='Train', color='blue')
        if 'val/dfl_loss' in training_df.columns:
            axes[0, 2].plot(training_df['epoch'], training_df['val/dfl_loss'], label='Val', color='red')
        axes[0, 2].set_xlabel('Epoch')
        axes[0, 2].set_ylabel('DFL Loss')
        axes[0, 2].set_title('Distribution Focal Loss')
        axes[0, 2].legend()
        axes[0, 2].grid(True)
    
    # Metrics
    if 'metrics/precision(B)' in training_df.columns:
        axes[1, 0].plot(training_df['epoch'], training_df['metrics/precision(B)'], label='Precision', color='green')
        if 'metrics/recall(B)' in training_df.columns:
            axes[1, 0].plot(training_df['epoch'], training_df['metrics/recall(B)'], label='Recall', color='orange')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Score')
        axes[1, 0].set_title('Precision & Recall')
        axes[1, 0].legend()
        axes[1, 0].grid(True)
    
    if 'metrics/mAP50(B)' in training_df.columns:
        axes[1, 1].plot(training_df['epoch'], training_df['metrics/mAP50(B)'], label='mAP@0.5', color='purple')
        if 'metrics/mAP50-95(B)' in training_df.columns:
            axes[1, 1].plot(training_df['epoch'], training_df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', color='brown')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('mAP')
        axes[1, 1].set_title('Mean Average Precision')
        axes[1, 1].legend()
        axes[1, 1].grid(True)
    
    # Learning rate
    if 'lr/pg0' in training_df.columns:
        axes[1, 2].plot(training_df['epoch'], training_df['lr/pg0'], color='teal')
        axes[1, 2].set_xlabel('Epoch')
        axes[1, 2].set_ylabel('Learning Rate')
        axes[1, 2].set_title('Learning Rate Schedule')
        axes[1, 2].grid(True)
    
    plt.suptitle('Training Progress', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(VIS_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nTraining curves saved to: {VIS_DIR / 'training_curves.png'}")

<Figure size 1800x1000 with 6 Axes>


Training curves saved to: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\visualizations\training_curves.png


## 7. Inference Examples

In [17]:
# Run inference on test images
test_images_path = DATASET_DIR / "test" / "images"

if test_images_path.exists():
    test_images = list(test_images_path.glob("*.jpg"))[:6]
    
    print(f"Running inference on {len(test_images)} test images...")
    
    # Run prediction
    predictions = best_model.predict(
        source=test_images,
        device=DEVICE,
        conf=0.25,  # Lower threshold to catch more detections
        save=True,
        project=str(VIS_DIR),
        name='inference_samples',
        exist_ok=True
    )
    
    print(f"\nInference results saved to: {VIS_DIR / 'inference_samples'}")
else:
    print(f"Test images not found at: {test_images_path}")

Running inference on 6 test images...

0: 640x640 1 Customer-Bagpack, 2 normals, 29.4ms
1: 640x640 1 Customer-Bagpack, 5 normals, 29.4ms
2: 640x640 1 Customer-Bagpack, 4 normals, 1 theft, 29.4ms
3: 640x640 1 Customer-Bagpack, 5 normals, 29.4ms
4: 640x640 4 Customer-Bagpacks, 5 normals, 1 theft, 29.4ms
5: 640x640 1 Customer-Bagpack, 4 normals, 29.4ms
Speed: 2.8ms preprocess, 29.4ms inference, 2.3ms postprocess per image at shape (1, 3, 640, 640)
Results saved to C:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\visualizations\inference_samples

Inference results saved to: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\visualizations\inference_samples


In [18]:
# Display inference results
import cv2

inference_dir = VIS_DIR / 'inference_samples'

if inference_dir.exists():
    result_images = list(inference_dir.glob("*.jpg"))
    
    if result_images:
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()
        
        for idx, img_path in enumerate(result_images[:6]):
            img = cv2.imread(str(img_path))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[idx].imshow(img)
            axes[idx].set_title(img_path.name[:30], fontsize=10)
            axes[idx].axis('off')
        
        # Hide empty subplots
        for idx in range(len(result_images), 6):
            axes[idx].axis('off')
        
        plt.suptitle('Inference Results on Test Images', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig(VIS_DIR / 'inference_results_grid.png', dpi=150, bbox_inches='tight')
        plt.show()

<Figure size 1800x1200 with 6 Axes>

## 8. Export Model

In [20]:
# Export model to various formats
print("\n" + "="*70)
print("MODEL EXPORT")
print("="*70)

export_dir = RUNS_DIR / 'retail_theft_yolov8s' / 'exports'
export_dir.mkdir(exist_ok=True)

# Export to ONNX (widely compatible)
print("\nExporting to ONNX format...")
onnx_path = best_model.export(format='onnx', imgsz=IMG_SIZE)
print(f"ONNX model saved to: {onnx_path}")

# Export to TorchScript
print("\nExporting to TorchScript format...")
torchscript_path = best_model.export(format='torchscript', imgsz=IMG_SIZE)
print(f"TorchScript model saved to: {torchscript_path}")


MODEL EXPORT

Exporting to ONNX format...
Ultralytics 8.3.247  Python-3.10.10 torch-2.9.1+cu130 CPU (Intel Core i5-10500H 2.50GHz)

PyTorch: starting from 'C:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\runs\retail_theft_yolov8s\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (21.5 MB)

ONNX: starting export with onnx 1.20.1 opset 22...
ONNX: slimming with onnxslim 0.1.82...
ONNX: export success  1.5s, saved as 'C:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\runs\retail_theft_yolov8s\weights\best.onnx' (42.7 MB)

Export complete (1.9s)
Results saved to C:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\runs\retail_theft_yolov8s\weights
Predict:         yolo predict task=detect model=C:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\runs\retail_theft_yolov8s\weights\best.onnx imgsz=640  
Validate:        yolo val task=detect model=C:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\

## 9. Save Final Report

In [21]:
# Generate final training report
final_report = {
    'timestamp': datetime.now().isoformat(),
    'model': 'YOLOv8s',
    'dataset': str(DATASET_DIR),
    'training_config': {
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'image_size': IMG_SIZE,
        'device': str(DEVICE),
        'early_stopping_patience': PATIENCE
    },
    'results': {
        'save_dir': str(results.save_dir),
        'best_model': str(best_model_path)
    },
    'validation_metrics': {
        'mAP50': float(val_results.box.map50),
        'mAP50_95': float(val_results.box.map),
        'precision': float(val_results.box.mp),
        'recall': float(val_results.box.mr)
    },
    'test_metrics': {
        'mAP50': float(test_results.box.map50),
        'mAP50_95': float(test_results.box.map),
        'precision': float(test_results.box.mp),
        'recall': float(test_results.box.mr)
    },
    'exports': {
        'onnx': str(onnx_path) if 'onnx_path' in dir() else None,
        'torchscript': str(torchscript_path) if 'torchscript_path' in dir() else None
    }
}

# Save report
report_path = OUTPUT_DIR / 'training_final_report.json'
with open(report_path, 'w') as f:
    json.dump(final_report, f, indent=2)

print(f"Final report saved to: {report_path}")

Final report saved to: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\outputs\training_final_report.json


In [22]:
# Print final summary
print("\n" + "="*70)
print("TRAINING AND EVALUATION COMPLETE")
print("="*70)

print(f"""
FINAL SUMMARY
=============

Model: YOLOv8s (Small)
Dataset: {DATASET_DIR.name}

VALIDATION METRICS:
  mAP@0.5:      {val_results.box.map50:.4f}
  mAP@0.5:0.95: {val_results.box.map:.4f}
  Precision:    {val_results.box.mp:.4f}
  Recall:       {val_results.box.mr:.4f}

TEST METRICS:
  mAP@0.5:      {test_results.box.map50:.4f}
  mAP@0.5:0.95: {test_results.box.map:.4f}
  Precision:    {test_results.box.mp:.4f}
  Recall:       {test_results.box.mr:.4f}

MODEL FILES:
  Best weights: {best_model_path}
  ONNX export:  {onnx_path if 'onnx_path' in dir() else 'N/A'}

REPORTS:
  Training curves: {VIS_DIR / 'training_curves.png'}
  Final report:    {report_path}

NEXT STEPS:
  1. Review per-class metrics, especially theft recall
  2. Test on real-world surveillance footage
  3. Fine-tune confidence threshold for optimal precision/recall
  4. Deploy using ONNX for production
""")

print("="*70)


TRAINING AND EVALUATION COMPLETE

FINAL SUMMARY

Model: YOLOv8s (Small)
Dataset: dataset_augmented

VALIDATION METRICS:
  mAP@0.5:      0.8427
  mAP@0.5:0.95: 0.6413
  Precision:    0.8370
  Recall:       0.7790

TEST METRICS:
  mAP@0.5:      0.7786
  mAP@0.5:0.95: 0.5764
  Precision:    0.7478
  Recall:       0.7475

MODEL FILES:
  Best weights: C:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\runs\retail_theft_yolov8s\weights\best.pt
  ONNX export:  C:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\runs\retail_theft_yolov8s\weights\best.onnx

REPORTS:
  Training curves: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\visualizations\training_curves.png
  Final report:    c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\outputs\training_final_report.json

NEXT STEPS:
  1. Review per-class metrics, especially theft recall
  2. Test on real-world surveillance footage
  3. Fine-tune confidence threshold for optimal precision/recall
  4.

---

## Troubleshooting

### Low Theft Class Recall
If theft class recall is below expectations:
1. Lower confidence threshold during inference (e.g., 0.2)
2. Apply more augmentation to theft samples
3. Fine-tune with higher cls_loss weight
4. Use focal loss with higher gamma

### Out of Memory (OOM) Errors
1. Reduce batch size
2. Disable mosaic augmentation (`mosaic=0.0`)
3. Use smaller image size (512 instead of 640)
4. Disable cache (`cache=False`)

### Slow Training
1. Verify GPU is being used (`device=0`)
2. Enable AMP (`amp=True`)
3. Use cache (`cache=True`)
4. Increase workers (`workers=8`)

---